# [Introduction to Joining Tables](https://learn.microsoft.com/en-us/training/modules/query-multiple-tables-with-joins/1-introduction) 

## Relational Databases and Normalization
Relational databases are built on the concept of **normalization**, a design process that organizes data into multiple tables to minimize data duplication and ensure data integrity. 

Instead of storing all information in one massive, flat table, data is split into logical, related tables (e.g., `Customers`, `Orders`, `Products`). These separate tables are linked together by **common key fields** (Primary Keys and Foreign Keys).

## Why We Need JOINs

Because data is distributed across multiple normalized tables, retrieving a complete business picture often requires combining data from two or more tables simultaneously. This is where **JOINs** come in.

A `JOIN` clause in SQL is used to combine rows from two or more tables based on a related column between them.

### Visualizing a JOIN

```mermaid
graph LR
    classDef table fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#0d47a1;
    classDef key fill:#fff3e0,stroke:#ef6c00,stroke-width:2px,color:#e65100;
    classDef join fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#1b5e20;
    classDef result fill:#f3e5f5,stroke:#6a1b9a,stroke-width:2px,color:#4a148c;

    subgraph CUSTOMERS["📋 Customers Table"]
        direction TB
        C1[("🔑 CustomerID (PK)")]:::key
        C2["👤 CustomerName"]:::table
        C3["📍 City"]:::table
    end

    subgraph ORDERS["📦 Orders Table"]
        direction TB
        O1[("🔑 OrderID (PK)")]:::key
        O2[("🔗 CustomerID (FK)")]:::key
        O3["📅 OrderDate"]:::table
    end

    subgraph JOIN["🔗 JOIN Operation"]
        direction TB
        J["SELECT 
        ────────────────
        Customers.CustomerName,
        Orders.OrderID,
        Orders.OrderDate
        ────────────────
        FROM Customers
        JOIN Orders
        ON Customers.CustomerID = Orders.CustomerID"]:::join
    end

    subgraph OUTPUT["📊 Result Set"]
        direction TB
        R1["CustomerName | OrderID | OrderDate"]:::result
        R2["Alice        | 101     | 2024-01-15"]:::result
        R3["Bob          | 102     | 2024-01-16"]:::result
        R4["Alice        | 103     | 2024-01-17"]:::result
    end

    CUSTOMERS -->|"CustomerID"| JOIN
    ORDERS -->|"CustomerID"| JOIN
    JOIN -->|"Combined Data"| OUTPUT
    
    C1 -.->|"Primary Key → Foreign Key"| O2
    
    style CUSTOMERS fill:#e3f2fd,stroke:#1565c0,stroke-width:2px
    style ORDERS fill:#e3f2fd,stroke:#1565c0,stroke-width:2px
    style JOIN fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px
    style OUTPUT fill:#f3e5f5,stroke:#6a1b9a,stroke-width:2px
```


***

## Module Learning Objectives

In this module, you will master the various techniques for combining data from multiple tables. By the end of this module, you will be able to:

1. **Understand join concepts and syntax**: Learn the foundational theory and standard SQL syntax for combining tables.
2. **Write queries that use inner joins**: Retrieve records that have matching values in both tables.
3. **Write queries that use outer joins**: Retrieve all records from one table and the matched records from the other (Left, Right, and Full Outer Joins).
4. **Write queries that use cross joins**: Generate a Cartesian product of two tables (every row in Table A matched with every row in Table B).
5. **Write queries that use self-joins**: Join a table to itself to compare or relate rows within the exact same table.


# [Understanding JOIN Concepts and Syntax](https://learn.microsoft.com/en-us/training/modules/query-multiple-tables-with-joins/2-join-concepts)


## The Role of the JOIN Operation
The most fundamental method of combining data from multiple tables is the **JOIN** operation. While some view `JOIN` as a separate clause, it is technically considered a part of the **`FROM`** clause in T-SQL. 

In this module, we will explore how the `FROM` clause creates intermediate **virtual tables** that are consumed by later phases of the query execution.
***

## The FROM Clause and Virtual Tables

If you understand the **logical order of operations** in SQL Server, you know that the `FROM` clause is the **very first clause processed**. 

### How it Works:
Understanding how SQL Server processes a query step-by-step is crucial.

1. **The First Step**: The `FROM` clause is the very first clause evaluated in a `SELECT` statement. It determines the source table(S) for the query.

2. **The Virtual Table**: The `FROM` clause takes the specified tables (single or joined) and creates a logical **Virtual Table** in memory. No physical table is created on disk.

3. **The Pipeline**: This virtual table acts as the raw data source that is passed downstream to subsequent clauses like `WHERE`, `GROUP BY`, and `SELECT`.

As you add JOINs to your query, imagine you are adding columns and rows to—or removing rows from—this intermediate virtual table.

### Logical Processing Flow
```mermaid
graph TD
    classDef from fill:#e3f2fd,stroke:#1565c0,stroke-width:2px;
    classDef virtual fill:#fff3e0,stroke:#ef6c00,stroke-width:2px;
    classDef where fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px;
    classDef select fill:#f3e5f5,stroke:#7b1fa2,stroke-width:2px;

    A["1. FROM Clause\n(Identifies base tables & applies JOINs)"]:::from 
    --> B["2. Virtual Table Created\n(Logical combination of all joined tables)"]:::virtual
    B --> C["3. WHERE Clause\n(Filters rows from the virtual table)"]:::where
    C --> D["4. SELECT Clause\n(Chooses columns to return)"]:::select
    
    note["The FROM clause's job is to add rows to or remove rows from the virtual table."]
    B -.-> note
```
***

## Conceptualizing Joins as Sets

It is highly useful to think of the results of a `FROM` clause as **sets** and visualize the join operations using **Venn diagrams**. 

When you join an `Employee` table to a `SalesOrder` table, you are essentially finding the intersection or combining the sets of data based on their relationship (e.g., an Employee *has* SalesOrders).

```mermaid
graph LR
    classDef table1 fill:#bbdefb,stroke:#1565c0,stroke-width:2px,color:#0d47a1;
    classDef table2 fill:#c8e6c9,stroke:#2e7d32,stroke-width:2px,color:#1b5e20;
    classDef link fill:#ffecb3,stroke:#ff8f00,stroke-width:2px,color:#e65100;
    classDef note fill:#f3e5f5,stroke:#6a1b9a,stroke-width:2px,stroke-dasharray: 5 5,color:#4a148c;

    subgraph EMPLOYEES["👨‍💼 Employees Set"]
        direction TB
        E1["Employee 1<br/>(EmpID: 1)"]:::table1
        E2["Employee 2<br/>(EmpID: 2)"]:::table1
        E3["Employee 3<br/>(EmpID: 3)"]:::table1
    end

    subgraph ORDERS["📦 SalesOrders Set"]
        direction TB
        O1["Order 101<br/>(EmpID: 1)"]:::table2
        O2["Order 102<br/>(EmpID: 1)"]:::table2
        O3["Order 103<br/>(EmpID: 2)"]:::table2
    end

    subgraph JOIN["🔗 JOIN Relationship"]
        direction TB
        J1["Employee 1 → Order 101"]:::link
        J2["Employee 1 → Order 102"]:::link
        J3["Employee 2 → Order 103"]:::link
    end

    E1 -->|"1 → Many"| J1
    E1 -->|"1 → Many"| J2
    E2 -->|"1 → Many"| J3
    
    J1 --> O1
    J2 --> O2
    J3 --> O3

    N["💡 A JOIN matches rows between sets<br/>based on a common key (EmpID)"]:::note
    
    EMPLOYEES -.-> N
    ORDERS -.-> N
    
    style EMPLOYEES fill:#bbdefb,stroke:#1565c0,stroke-width:2px
    style ORDERS fill:#c8e6c9,stroke:#2e7d32,stroke-width:2px
    style JOIN fill:#ffecb3,stroke:#ff8f00,stroke-width:2px
    style N fill:#f3e5f5,stroke:#6a1b9a,stroke-width:2px,stroke-dasharray: 5 5
```

***

## Evolution of JOIN Syntax: SQL-89 vs. SQL-92

Throughout its history, T-SQL has evolved to match ANSI standards. The syntax for joins has changed significantly between the older SQL-89 standard and the modern SQL-92 standard.

### 1. ANSI SQL-89 (Legacy Syntax - Not Recommended)
In the older standard, multiple tables were listed in the `FROM` clause separated by commas. The join condition was hidden inside the `WHERE` clause.

```sql
-- ANSI SQL-89 Style
SELECT p.ProductID, m.Name AS Model, p.Name AS Product
FROM SalesLT.Product AS p, SalesLT.ProductModel AS m
WHERE p.ProductModelID = m.ProductModelID;
```
*⚠️ **Danger**: If you forget the `WHERE` clause, the query silently runs and creates a massive Cartesian product!*

### 2. ANSI SQL-92 (Modern Syntax - Highly Recommended)
The modern standard introduced explicit `JOIN` and `ON` keywords directly in the `FROM` clause.

```sql
-- ANSI SQL-92 Style
SELECT p.ProductID, m.Name AS Model, p.Name AS Product
FROM SalesLT.Product AS p
JOIN SalesLT.ProductModel AS m
    ON p.ProductModelID = m.ProductModelID;
```
✅ **Benefit**: If you forget the `ON` clause, SQL Server immediately throws a **syntax error**, preventing accidental Cartesian products.

***

**Example: Comparing Syntax**
```sql
-- ❌ LEGACY (SQL-89): Comma separation, filtering in WHERE
SELECT p.ProductID, m.Name AS Model, p.Name AS Product
FROM SalesLT.Product AS p, SalesLT.ProductModel AS m
WHERE p.ProductModelID = m.ProductModelID;

-- ✅ MODERN (SQL-92): Explicit JOIN and ON clause
SELECT p.ProductID, m.Name AS Model, p.Name AS Product
FROM SalesLT.Product AS p
JOIN SalesLT.ProductModel AS m
    ON p.ProductModelID = m.ProductModelID;
```

***

## The Danger of Cartesian Products

A **Cartesian product** occurs when two tables are joined without any relationship or join condition. Every single row in Table A is combined with every single row in Table B.

### The Math Behind It:
If Table A has **2 rows** and Table B has **6 rows**, the Cartesian product will yield **12 rows** (2 x 6 = 12). 
In a database, if you join a table with **10 rows** to a table with **100 rows** without an `ON` clause, you get **1,000 rows**.

### Visualizing a Cartesian Product
```mermaid
flowchart TD
    classDef table fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#0d47a1;
    classDef result fill:#ffebee,stroke:#c62828,stroke-width:2px,color:#b71c1c;
    classDef highlight fill:#fff3e0,stroke:#ef6c00,stroke-width:2px,color:#e65100;

    subgraph TABLEA["📋 Table A: Names (2 rows)"]
        direction TB
        A1["👤 Alice"]:::table
        A2["👤 Bob"]:::table
    end

    subgraph TABLEB["📦 Table B: Products (3 rows)"]
        direction TB
        B1["🚲 Bike"]:::table
        B2["⛑️ Helmet"]:::table
        B3["🧤 Gloves"]:::table
    end

    subgraph RESULT["🔄 Cartesian Product: 2 × 3 = 6 rows"]
        direction TB
        C1["Alice - Bike"]:::result
        C2["Alice - Helmet"]:::result
        C3["Alice - Gloves"]:::result
        C4["Bob - Bike"]:::result
        C5["Bob - Helmet"]:::result
        C6["Bob - Gloves"]:::result
    end

    A1 --> C1
    A1 --> C2
    A1 --> C3
    A2 --> C4
    A2 --> C5
    A2 --> C6
    
    C1 --> B1
    C2 --> B2
    C3 --> B3
    C4 --> B1
    C5 --> B2
    C6 --> B3

    NOTE["💡 All rows from Table A paired with all rows from Table B"]:::highlight
    
    TABLEA -.-> NOTE
    TABLEB -.-> NOTE
    
    style TABLEA fill:#e3f2fd,stroke:#1565c0,stroke-width:2px
    style TABLEB fill:#e3f2fd,stroke:#1565c0,stroke-width:2px
    style RESULT fill:#ffebee,stroke:#c62828,stroke-width:2px
    style NOTE fill:#fff3e0,stroke:#ef6c00,stroke-width:2px,stroke-dasharray: 5 5
```

### Why is this bad?
While useful for generating test data, accidental Cartesian products in production queries will:
1. Return completely **incorrect business results**.
2. Cause **severe performance issues** and memory exhaustion due to the massive result set.

***

## Summary

- The `FROM` clause is processed first and creates a **logical virtual table**.
- Always use the **ANSI SQL-92 syntax** (`JOIN ... ON ...`) rather than the legacy SQL-89 comma-separated syntax.
- The modern syntax protects you from accidentally creating **Cartesian products** (where every row in one table matches every row in another) by forcing you to define the `ON` relationship.
- Conceptualize your joins using **Venn diagrams** to understand how sets of data are being combined or filtered.


# [Using INNER JOINs](https://learn.microsoft.com/en-us/training/modules/query-multiple-tables-with-joins/3a-inner-joins)

## What is an INNER JOIN?
The `INNER JOIN` is the most frequent type of join used in T-SQL. In a highly normalized database, data is scattered across multiple tables; the `INNER JOIN` is the primary tool used to stitch that related data back together to answer business questions.

An `INNER JOIN` returns **only the rows that have matching values** in both tables based on the predicate you define.


### How it Works Logically
Conceptually, an `INNER JOIN` begins its logical processing phase by creating a **Cartesian product** of the two tables. It then applies the `ON` predicate to **filter out** any rows that do not match, leaving only the rows where the join condition is met in both tables.

> **Note:** While SQL Server logically processes it as a Cartesian product followed by a filter, the physical execution engine (Query Optimizer) will use much more efficient algorithms (like Hash Match or Merge Join) under the hood.

***

## Step-by-Step Logical Processing

Let's track how SQL Server logically processes a basic `INNER JOIN` query:

```sql
1) SELECT emp.FirstName, ord.Amount
2) FROM HR.Employee AS emp 
3) JOIN Sales.SalesOrder AS ord
4)      ON emp.EmployeeID = ord.EmployeeID;
```

### Execution Flow
```mermaid
graph TD
    A[(Table 1: HR.Employee)] -->|FROM| C
    B[(Table 2: Sales.SalesOrder)] -->|JOIN| C
    C[[Logical Cartesian Product <br> Every Employee combined with Every Order]] --> D{ON Clause Applied <br> emp.EmployeeID = ord.EmployeeID}
    D -->|Match Found| E([Keep Row in Virtual Table])
    D -->|No Match| F[Discard Row]
    
    style C fill:#fff3e0,stroke:#f57c00,stroke-width:2px
    style E fill:#d4edda,stroke:#2e7d32,stroke-width:2px
    style F fill:#f8d7da,stroke:#c62828,stroke-width:2px
```
### Step-by-Step Breakdown

1. `FROM / JOIN`: The engine identifies the input tables. Logically, a Cartesian product is generated (all possible combinations).

2. `ON`: The engine immediately filters this virtual table, keeping only the rows where the specified condition evaluates to TRUE (e.g., the EmployeeID matches in both tables).

3. **Unmatched Data Dropped**: Employees with zero orders are filtered out. Orders with invalid or missing Employee IDs are filtered out.

4. `SELECT`: The engine extracts only the requested columns from the remaining matched rows.

**Result:** Only employees who have associated orders, and only orders that belong to a valid employee, are returned.

***

## Visualizing the INNER JOIN Result

An `INNER JOIN` returns only the **intersection** of the two tables based on the join condition. Rows that exist in one table but not the other are completely excluded from the final result.

```mermaid
graph TD
    classDef match fill:#c8e6c9,stroke:#2e7d32,stroke-width:2px;
    classDef noMatch fill:#ffcdd2,stroke:#c62828,stroke-width:2px;
    
    subgraph HR.Employee
        E1["Emp 1\n(Has Orders)"]:::match
        E2["Emp 2\n(No Orders)"]:::noMatch
        E3["Emp 3\n(Has Orders)"]:::match
    end
    
    subgraph Sales.SalesOrder
        O1["Ord 101\n(Valid Emp)"]:::match
        O2["Ord 102\n(Valid Emp)"]:::match
        O3["Ord 103\n(Invalid Emp ID)"]:::noMatch
    end
    
    E1 <-->|Match| O1
    E1 <-->|Match| O2
    E3 <-->|Match| O1
    
    note["<b>INNER JOIN Result:</b><br>Only the green 'matching' rows are returned.<br>Emp 2 and Ord 103 are excluded."]
```

***

## INNER JOIN Syntax and Guidelines

### Syntax
The `INNER` keyword is actually optional in T-SQL. If you just write `JOIN`, SQL Server assumes `INNER JOIN`. However, it is a best practice to write `INNER JOIN` explicitly for readability.

```sql
-- Both are valid, but the second is preferred for clarity
SELECT ... FROM TableA JOIN TableB ON ...
SELECT ... FROM TableA INNER JOIN TableB ON ...
```

### Best Practices & Guidelines
1. **Use Table Aliases**: Always use aliases (e.g., `AS emp`) not just in the `SELECT` list, but especially in the `ON` clause to avoid ambiguity and save typing.
2. **Composite Joins**: You can join on a single column (e.g., `OrderID`) or multiple columns (e.g., `OrderID` AND `ProductID`).
3. **Table Order**: The order in which you list tables in the `FROM` clause **does not matter** to the SQL Server Query Optimizer. It will figure out the most efficient physical execution path.
4. **One JOIN per pair**: For every additional table you add, you must use the `JOIN ... ON ...` syntax once.

***

**Basic INNER JOIN Example**
```sql
-- Joining two tables on a single matching column (ProductModelID)
-- Notice the use of table aliases (p and m) for clarity.

SELECT p.ProductID, 
       m.Name AS Model, 
       p.Name AS Product
FROM Production.Product AS p
INNER JOIN Production.ProductModel AS m
    ON p.ProductModelID = m.ProductModelID
ORDER BY p.ProductID;
```

***

## Chaining Multiple INNER JOINs

You can easily extend an `INNER JOIN` to include three or more tables. Each `JOIN ... ON ...` block does its own population and filtering of the virtual output table.

### How it Works:
1. Table 1 and Table 2 are joined and filtered.
2. The resulting virtual table is then joined to Table 3 and filtered again.
3. The final virtual table is passed to the `SELECT` clause.

*Note: The SQL Server Query Optimizer determines the actual physical order in which these joins are executed to maximize performance.*

***

**Multiple INNER JOINs Example**
```sql
-- Joining THREE tables together
-- 1. Product (p) joins to ProductModel (m)
-- 2. That result joins to SalesOrderDetail (od)

SELECT od.SalesOrderID, 
       m.Name AS Model, 
       p.Name AS ProductName, 
       od.OrderQty
FROM Production.Product AS p
INNER JOIN Production.ProductModel AS m
    ON p.ProductModelID = m.ProductModelID
INNER JOIN Sales.SalesOrderDetail AS od
    ON p.ProductID = od.ProductID
ORDER BY od.SalesOrderID;
```

***

## Summary

- **`INNER JOIN`** returns only the rows that have matching values in **both** joined tables.
- Logically, it creates a Cartesian product and then filters it using the `ON` predicate.
- The `INNER` keyword is optional but recommended for explicit clarity.
- Always use **table aliases** to make your queries cleaner and prevent column ambiguity.
- You can chain multiple `INNER JOIN`s to combine data from three or more tables.
- The physical execution order of tables is determined by the Query Optimizer, not the order you write them in the `FROM` clause.



# [Using OUTER JOINs](https://learn.microsoft.com/en-us/training/modules/query-multiple-tables-with-joins/3b-outer-joins)

## What is an OUTER JOIN?
While `INNER JOIN` only returns rows that have matching values in **both** tables, an **`OUTER JOIN`** provides a broader view of your business data. 

An `OUTER JOIN` retrieves:
1. All rows with matching attributes between the tables.
2. **All rows** present in one (or both) of the tables, **even if there is no match** in the other table.

When a row from the preserved table has no match in the other table, the columns from the unmatched table will return **`NULL`** values.

***

## INNER JOIN vs. LEFT OUTER JOIN

Let's compare how an `INNER JOIN` and a `LEFT OUTER JOIN` process the same data.

### The Scenario
We want to see Employees and their Sales Orders.
- **Employee 1** has orders.
- **Employee 2** has **no** orders.

### Logical Processing Comparison

```mermaid
flowchart TD
    classDef emp fill:#bbdefb,stroke:#1565c0,stroke-width:2px,color:#0d47a1;
    classDef ord fill:#c8e6c9,stroke:#2e7d32,stroke-width:2px,color:#1b5e20;
    classDef match fill:#fff9c4,stroke:#f57f17,stroke-width:2px,color:#e65100;
    classDef nullVal fill:#f5f5f5,stroke:#9e9e9e,stroke-width:2px,stroke-dasharray: 5 5,color:#616161;
    classDef drop fill:#ffebee,stroke:#c62828,stroke-width:2px,color:#b71c1c;
    classDef label fill:none,stroke:none,color:#333;

    subgraph INPUT["📥 Input Data"]
        direction TB
        E1["👨‍💼 Emp 1"]:::emp
        E2["👨‍💼 Emp 2"]:::emp
        O1["📦 Ord 101"]:::ord
    end

    subgraph INNER["🔗 INNER JOIN Result"]
        direction TB
        I1["Emp 1 → Ord 101 ✅"]:::match
        I2["Emp 2 → DROPPED ❌"]:::drop
    end

    subgraph LEFT["🔗 LEFT OUTER JOIN Result"]
        direction TB
        L1["Emp 1 → Ord 101 ✅"]:::match
        L2["Emp 2 → NULL ⚠️"]:::nullVal
    end

    E1 -->|"Has Order"| I1
    E1 -->|"Has Order"| L1
    E2 -->|"No Order"| I2
    E2 -->|"No Order"| L2
    O1 -->|"Belongs to Emp 1"| I1
    O1 -->|"Belongs to Emp 1"| L1

    NOTE1["💡 INNER JOIN: Only rows with matches in BOTH tables"]:::label
    NOTE2["💡 LEFT JOIN: All rows from LEFT table + matches from RIGHT"]:::label
    
    INNER -.-> NOTE1
    LEFT -.-> NOTE2
    
    style INPUT fill:#f5f5f5,stroke:#9e9e9e,stroke-width:2px
    style INNER fill:#fff3e0,stroke:#ef6c00,stroke-width:2px
    style LEFT fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px
```

**Key Takeaway:** The `LEFT OUTER JOIN` preserves **Employee 2**, filling the missing Order data with `NULL`.

***

**LEFT OUTER JOIN Example**
```sql
-- INNER JOIN: Only returns employees who HAVE placed orders
SELECT emp.FirstName, ord.Amount
FROM HR.Employee AS emp
INNER JOIN Sales.SalesOrder AS ord
    ON emp.EmployeeID = ord.EmployeeID;

-- LEFT OUTER JOIN: Returns ALL employees, even those with NO orders
-- Employees without orders will show NULL for the Amount
SELECT emp.FirstName, ord.Amount
FROM HR.Employee AS emp
LEFT OUTER JOIN Sales.SalesOrder AS ord
    ON emp.EmployeeID = ord.EmployeeID;
```

***

## OUTER JOIN Syntax and Types

Outer joins are expressed using the keywords **`LEFT`**, **`RIGHT`**, or **`FULL`** preceding `OUTER JOIN`. 

> **Note:** The word `OUTER` is actually optional in T-SQL. You can just write `LEFT JOIN`, `RIGHT JOIN`, or `FULL JOIN`. However, writing `LEFT OUTER JOIN` is a best practice for explicit clarity.

### Visualizing the Types

```mermaid
graph TD
    classDef left fill:#bbdefb,stroke:#1565c0,stroke-width:2px,color:#0d47a1;
    classDef right fill:#c8e6c9,stroke:#2e7d32,stroke-width:2px,color:#1b5e20;
    classDef both fill:#fff9c4,stroke:#f57f17,stroke-width:2px,color:#e65100;
    classDef highlight fill:#fff3e0,stroke:#ef6c00,stroke-width:2px,color:#e65100;

    subgraph LEFTJOIN["🔗 LEFT OUTER JOIN"]
        direction TB
        L1["👈 All Rows from Left Table"]:::left
        L2["➕ Matching Rows from Right"]:::both
    end

    subgraph RIGHTJOIN["🔗 RIGHT OUTER JOIN"]
        direction TB
        R1["👉 All Rows from Right Table"]:::right
        R2["➕ Matching Rows from Left"]:::both
    end

    subgraph FULLJOIN["🔗 FULL OUTER JOIN"]
        direction TB
        F1["👈 All Rows from Left Table"]:::left
        F2["👉 All Rows from Right Table"]:::right
        F3["➕ Matching Rows from Both"]:::both
    end

    NOTE["💡 OUTER JOINS preserve unmatched rows"]:::highlight
    
    LEFTJOIN -.-> NOTE
    RIGHTJOIN -.-> NOTE
    FULLJOIN -.-> NOTE
    
    style LEFTJOIN fill:#e3f2fd,stroke:#1565c0,stroke-width:2px
    style RIGHTJOIN fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px
    style FULLJOIN fill:#f3e5f5,stroke:#6a1b9a,stroke-width:2px
    style NOTE fill:#fff3e0,stroke:#ef6c00,stroke-width:2px,stroke-dasharray: 5 5
```
```mermaid
flowchart TD
    A["JOIN Types"] --> B["LEFT JOIN"]
    A --> C["RIGHT JOIN"]
    A --> D["FULL JOIN"]
    
    B --> B1["Preserves Left Table"]
    B --> B2["NULL for unmatched Right"]
    
    C --> C1["Preserves Right Table"]
    C --> C2["NULL for unmatched Left"]
    
    D --> D1["Preserves Both Tables"]
    D --> D2["NULL for unmatched rows"]
    
    style B fill:#e3f2fd,stroke:#1565c0
    style C fill:#e8f5e9,stroke:#2e7d32
    style D fill:#f3e5f5,stroke:#6a1b9a
```
***

**RIGHT and FULL OUTER JOIN Examples**
```sql
-- RIGHT OUTER JOIN: Preserves ALL rows from the Right table (SalesOrder)
-- Useful if you want to see all orders, even if the Employee record is missing/broken
SELECT emp.FirstName, ord.Amount
FROM HR.Employee AS emp
RIGHT OUTER JOIN Sales.SalesOrder AS ord
    ON emp.EmployeeID = ord.EmployeeID;

-- FULL OUTER JOIN: Preserves ALL rows from BOTH tables
-- Returns matches, plus unmatched employees, plus unmatched orders
SELECT emp.FirstName, ord.Amount
FROM HR.Employee AS emp
FULL OUTER JOIN Sales.SalesOrder AS ord
    ON emp.EmployeeID = ord.EmployeeID;
```

***

## Important Guidelines & Best Practices for OUTER JOINs

When writing queries with `OUTER JOIN`, keep these critical rules in mind:

### 1. Table Order Matters!
Unlike `INNER JOIN`, the order of tables in the `FROM` clause **dictates the logic**. 
- The table listed **first** (on the left of the `JOIN` keyword) is the **Left** table.
- The table listed **second** (on the right) is the **Right** table.

### 2. Finding "Non-Matches" (Anti-Joins)
To find rows in the left table that **do not** have a match in the right table, combine a `LEFT JOIN` with a `WHERE ... IS NULL` clause.

### 3. Multi-Table JOIN Complexity
Be very careful when chaining an `OUTER JOIN` to a third table. If the first `OUTER JOIN` produces `NULL` values, and your second `JOIN` uses an `INNER JOIN` on those `NULL` columns, **the unmatched rows will be filtered out**, effectively turning your `OUTER JOIN` back into an `INNER JOIN`!

### 4. Always use ORDER BY
Because the query optimizer processes outer joins differently than inner joins, there is no guaranteed order of results. Always use an `ORDER BY` clause if you need predictable output.

***

**Finding Non-Matches & Multi-Table Warnings**
```sql
-- ==========================================
-- FINDING NON-MATCHES (Employees with NO orders)
-- ==========================================
SELECT emp.FirstName, ord.Amount
FROM HR.Employee AS emp
LEFT OUTER JOIN Sales.SalesOrder AS ord
    ON emp.EmployeeID = ord.EmployeeID
WHERE ord.SalesOrderID IS NULL; -- Filters to keep ONLY the unmatched rows


-- ==========================================
-- MULTI-TABLE JOIN WARNING
-- ==========================================
-- ❌ BAD: The INNER JOIN to the 3rd table will filter out the NULLs 
-- generated by the LEFT JOIN, ruining the outer join!
SELECT emp.FirstName, ord.Amount, c.CompanyName
FROM HR.Employee AS emp
LEFT OUTER JOIN Sales.SalesOrder AS ord
    ON emp.EmployeeID = ord.EmployeeID
INNER JOIN Sales.Customer AS c -- This INNER JOIN kills the LEFT JOIN!
    ON ord.CustomerID = c.CustomerID;

-- ✅ GOOD: Use LEFT JOIN for all tables in the chain to preserve the outer rows
SELECT emp.FirstName, ord.Amount, c.CompanyName
FROM HR.Employee AS emp
LEFT OUTER JOIN Sales.SalesOrder AS ord
    ON emp.EmployeeID = ord.EmployeeID
LEFT OUTER JOIN Sales.Customer AS c 
    ON ord.CustomerID = c.CustomerID;
```

***

## Summary

- **`OUTER JOIN`** returns matching rows **plus** all rows from one or both tables, filling missing data with `NULL`.
- **`LEFT OUTER JOIN`**: Preserves all rows from the first (left) table.
- **`RIGHT OUTER JOIN`**: Preserves all rows from the second (right) table.
- **`FULL OUTER JOIN`**: Preserves all rows from both tables (used rarely).
- **Table order matters**: It determines which table is "Left" and which is "Right".
- **Finding non-matches**: Use `LEFT JOIN` combined with `WHERE <right_table_key> IS NULL`.
- **Multi-table caution**: Mixing `INNER JOIN` and `OUTER JOIN` in a chain can accidentally filter out your `NULL` values. Keep the chain consistent (e.g., all `LEFT JOIN`s) if you want to preserve the outer rows.
- Always use **`ORDER BY`** to guarantee the order of your results.



# Using CROSS JOINs

## What is a CROSS JOIN?
A **`CROSS JOIN`** is an explicit Cartesian product of two tables. It combines **every single row** in the first table with **every single row** in the second table, without any filtering or matching condition.

### The Math:
If Table A has **`m`** rows and Table B has **`n`** rows, a `CROSS JOIN` will produce **`m × n`** rows in the result set.

> ⚠️ **Warning:** Cross joins are rarely the desired output in production queries. They can easily generate massive, unmanageable result sets and cause severe performance problems. However, they do have a few valuable practical applications.

## Visualizing a CROSS JOIN

A `CROSS JOIN` creates every possible combination of rows between two tables. There is **no relationship** or matching condition applied.

```mermaid
flowchart TD
    classDef table1 fill:#bbdefb,stroke:#1565c0,stroke-width:2px,color:#0d47a1;
    classDef table2 fill:#c8e6c9,stroke:#2e7d32,stroke-width:2px,color:#1b5e20;
    classDef result fill:#fff9c4,stroke:#f57f17,stroke-width:2px,color:#e65100;
    classDef highlight fill:#fff3e0,stroke:#ef6c00,stroke-width:2px,stroke-dasharray:5 5;

    subgraph TABLEA["📋 Table A: Employees (3 rows)"]
        direction TB
        A1["👤 Alice"]:::table1
        A2["👤 Bob"]:::table1
        A3["👤 Carol"]:::table1
    end

    subgraph TABLEB["📦 Table B: Products (2 rows)"]
        direction TB
        B1["🚲 Bike"]:::table2
        B2["⛑️ Helmet"]:::table2
    end

    subgraph RESULT["🔄 CROSS JOIN Result: 3 × 2 = 6 rows"]
        direction TB
        R1["Alice → Bike ✅"]:::result
        R2["Alice → Helmet ✅"]:::result
        R3["Bob → Bike ✅"]:::result
        R4["Bob → Helmet ✅"]:::result
        R5["Carol → Bike ✅"]:::result
        R6["Carol → Helmet ✅"]:::result
    end

    A1 -->|"All combinations"| R1
    A1 -->|"..."| R2
    A2 -->|"All combinations"| R3
    A2 -->|"..."| R4
    A3 -->|"All combinations"| R5
    A3 -->|"..."| R6
    
    R1 --> B1
    R2 --> B2
    R3 --> B1
    R4 --> B2
    R5 --> B1
    R6 --> B2

    NOTE["💡 Every row from Table A is paired with every row from Table B"]:::highlight
    
    TABLEA -.-> NOTE
    TABLEB -.-> NOTE
    
    style TABLEA fill:#bbdefb,stroke:#1565c0,stroke-width:2px
    style TABLEB fill:#c8e6c9,stroke:#2e7d32,stroke-width:2px
    style RESULT fill:#fff9c4,stroke:#f57f17,stroke-width:2px
    style NOTE fill:#fff3e0,stroke:#ef6c00,stroke-width:2px,stroke-dasharray:5 5
```


***

## CROSS JOIN Syntax: SQL-89 vs SQL-92

### ANSI SQL-89 (Legacy - Accidental Cross Joins)
In the older syntax, a cross join happens **accidentally** when you list two tables in the `FROM` clause but forget the `WHERE` clause that links them. This is one of the main reasons the SQL-89 syntax is discouraged.

### ANSI SQL-92 (Modern - Explicit)
In the modern syntax, you must explicitly use the `CROSS JOIN` keyword. This makes it much harder to create an accidental Cartesian product.

### Syntax Rules
| Rule | Detail |
| :--- | :--- |
| **No `ON` Clause** | A `CROSS JOIN` performs no row matching. Using an `ON` clause with `CROSS JOIN` will cause a syntax error. |
| **Explicit Keyword** | Use the `CROSS JOIN` operator between the two table names. |
| **Table Aliases** | As always, use aliases for readability. |

```sql
-- ==========================================
-- MODERN SYNTAX (ANSI SQL-92) - RECOMMENDED
-- ==========================================
-- Explicit CROSS JOIN: All combinations of employees and products
SELECT emp.FirstName, prd.Name
FROM HR.Employee AS emp
CROSS JOIN Production.Product AS prd;


-- ==========================================
-- LEGACY SYNTAX (ANSI SQL-89) - AVOID
-- ==========================================
-- This creates the same Cartesian product, but it's unclear 
-- whether the missing WHERE clause was intentional or a mistake!
SELECT emp.FirstName, prd.Name
FROM HR.Employee AS emp, Production.Product AS prd;
-- (No WHERE clause = accidental cross join risk)
```

## When Should You Actually Use a CROSS JOIN?

While cross joins are usually unwanted, there are a few legitimate use cases:

### 1. 🔢 Creating a Table of Numbers
You can use a `CROSS JOIN` to generate a sequential range of numbers, which is useful for reporting gaps or generating date ranges.

### 2. 🧪 Generating Test Data
A `CROSS JOIN` is an incredibly fast way to generate large volumes of test data. When you cross join a table **to itself**, a table with just **100 rows** will instantly generate **10,000 rows** (100 × 100).

### Scaling Impact Visualization
```mermaid
graph LR
    classDef small fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px;
    classDef medium fill:#fff3e0,stroke:#ef6c00,stroke-width:2px;
    classDef large fill:#ffebee,stroke:#c62828,stroke-width:2px;

    A["10 rows × 10 rows\n= 100 rows"]:::small
    B["100 rows × 100 rows\n= 10,000 rows"]:::medium
    C["1,000 rows × 1,000 rows\n= 1,000,000 rows"]:::large

    A -->|"Grows exponentially!"| B --> C
```

> ⚠️ Caution: Be extremely careful with large tables. A cross join between two tables with 10,000 rows each will produce 100 million rows, which can crash your server or timeout your query.


***

**Practical CROSS JOIN Example**
```sql
-- ==========================================
-- PRACTICAL EXAMPLE: Generating Test Data
-- ==========================================
-- Cross joining a small table to itself to quickly 
-- generate a large dataset for testing purposes.

-- If this table has 100 rows, this query returns 10,000 rows
SELECT t1.SomeColumn AS Col1, 
       t2.SomeColumn AS Col2
FROM dbo.TestTable AS t1
CROSS JOIN dbo.TestTable AS t2;


-- ==========================================
-- PRACTICAL EXAMPLE: Generating Combinations
-- ==========================================
-- Creating all possible combinations of Size and Color 
-- (useful for a product variant matrix)

SELECT s.SizeName, c.ColorName
FROM dbo.Sizes AS s
CROSS JOIN dbo.Colors AS c;
```

## Summary

- A **`CROSS JOIN`** creates a Cartesian product: every row in Table A combined with every row in Table B.
- **No `ON` clause** is used (and using one will cause a syntax error).
- The modern **ANSI SQL-92** syntax (`CROSS JOIN`) is preferred because it makes the intent explicit and prevents accidental Cartesian products.
- The result set size grows **multiplicatively** (`m × n`), which can quickly overwhelm your system with large tables.
- **Legitimate uses** include generating number tables, creating all-possible-combination matrices, and generating large volumes of test data.
- In almost all other cases, if you see a cross join in your results, it means you have a **bug** in your query (usually a missing join condition).


# Using Self Joins

## What is a Self Join?
A **self-join** is not a distinct join type (like `INNER` or `LEFT`), but rather a powerful technique where a table is joined to **itself**. 

You use a self-join when you need to retrieve and compare rows from a table with other rows within that exact same table. 

### Common Scenarios:
1. **Hierarchical Data**: An Employee table where employees report to managers, and those managers are *also* listed as employees in the same table.
2. **Row Comparison**: Comparing values across different rows in the same table (e.g., matching a "Start Time" row with a "Stop Time" row for the same process).

## The Classic Scenario: Employee-Manager Hierarchy

Imagine an `HR.Employee` table. Every employee has an `EmployeeID`, and a `ManagerID` that points to the `EmployeeID` of their boss. Because the manager is also an employee, both the subordinate and the manager exist in the exact same table.

### Visualizing the Hierarchy
```mermaid
graph TD
    classDef boss fill:#ffecb3,stroke:#ff8f00,stroke-width:2px;
    classDef emp fill:#e3f2fd,stroke:#1565c0,stroke-width:2px;

    Dan["<b>Dan</b> (ID: 1)<br>Manager: NULL"]:::boss
    Aisha["<b>Aisha</b> (ID: 2)<br>Manager: Dan (1)"]:::emp
    Rosie["<b>Rosie</b> (ID: 3)<br>Manager: Dan (1)"]:::emp
    Naomi["<b>Naomi</b> (ID: 4)<br>Manager: Rosie (3)"]:::emp

    Dan --> Aisha
    Dan --> Rosie
    Rosie --> Naomi
    
    note["To see the Employee Name AND the Manager Name side-by-side,<br>we must query the table twice!"]
```


***

**Example: Writing a Self Join**
```sql
-- To join a table to itself, we MUST use table aliases 
-- to treat them as two separate logical tables (emp and mgr).

SELECT emp.FirstName AS Employee, 
       mgr.FirstName AS Manager
FROM HR.Employee AS emp
LEFT OUTER JOIN HR.Employee AS mgr
    ON emp.ManagerID = mgr.EmployeeID;
```

## Why use a LEFT OUTER JOIN for Self Joins?

Notice that in the previous query, we used a **`LEFT OUTER JOIN`** instead of an `INNER JOIN`. 

### The Reason:
The CEO (or top-level boss) does not have a manager. Their `ManagerID` is `NULL`. 
- If we used an `INNER JOIN`, the CEO would be completely excluded from the results because there is no matching `EmployeeID` for a `NULL` `ManagerID`.
- By using a `LEFT OUTER JOIN`, we preserve **all** employees in the left table (`emp`), and the query simply returns `NULL` for the manager's name when a match isn't found.

### Expected Output:
| Employee | Manager |
| :--- | :--- |
| Dan | *NULL* |
| Aisha | Dan |
| Rosie | Dan |
| Naomi | Rosie |

## Best Practices & Guidelines

When writing self-joins, follow these critical rules:

1. **Define Two Instances**: You must list the same table twice in the `FROM` clause.
2. **Always Use Aliases**: You **must** assign different aliases to each instance (e.g., `AS emp` and `AS mgr`). If you don't, SQL Server won't know which "version" of the table you are referring to, and the query will fail.
3. **Use the ON Clause**: Use the `ON` clause to explicitly define how the columns from the first instance relate to the columns in the second instance.
4. **Choose the Right Join Type**: Decide if you need an `INNER JOIN` (only rows with a match) or an `OUTER JOIN` (preserve rows even if the relationship is missing, like the CEO).

```sql
-- ==========================================
-- SCENARIO: Comparing different rows in the same table
-- ==========================================
-- Finding employees who have the same manager as another employee,
-- or comparing a "Start" event to an "End" event in an audit log.

-- Example: Find pairs of employees who report to the SAME manager
SELECT e1.FirstName AS Employee1, 
       e2.FirstName AS Employee2, 
       e1.ManagerID
FROM HR.Employee AS e1
INNER JOIN HR.Employee AS e2
    ON e1.ManagerID = e2.ManagerID
    AND e1.EmployeeID < e2.EmployeeID; -- Prevents matching an employee to themselves and avoids duplicate pairs
```
## Summary

- A **Self Join** is a technique where a table is joined to itself to compare or relate rows within the same table.
- It is most commonly used for **hierarchical data** (e.g., Employee/Manager relationships).
- **Table aliases are mandatory** to differentiate the two instances of the same table.
- Use a **`LEFT OUTER JOIN`** when you want to include rows that do not have a matching relationship in the same table (like a CEO with no manager).
- Self joins can also be used to compare data across different rows, such as matching start and stop events in a single audit table.
